In [1]:
from langchain_ollama import OllamaLLM
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [9]:
from langchain_core.messages import BaseMessage,HumanMessage,SystemMessage,AIMessage
from langgraph.graph import END,MessageGraph,StateGraph
from typing import List,Sequence
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

<div style="text-align: center;">
  <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Cbuc3z8N1_Ew2ESw199Slw/Workflow.png" alt="Workflow" style="width: 40%; height: 500px;">
</div>


#### Generation Prompt for posts

In [3]:
generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a professional LinkedIn content assistant tasked with crafting engaging, insightful, and well-structured LinkedIn posts."
            " Generate the best LinkedIn post possible for the user's request."
            " If the user provides feedback or critique, respond with a refined version of your previous attempts, improving clarity, tone, or engagement as needed.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [4]:
generation_chain=generation_prompt|llm

##### Reflection Prompt for linkedin post critique

In [7]:
reflection_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a professional LinkedIn content strategist and thought leadership expert. Your task is to critically evaluate the given LinkedIn post and provide a comprehensive critique. Follow these guidelines:

        1. Assess the posts overall quality, professionalism, and alignment with LinkedIn best practices.
        2. Evaluate the structure, tone, clarity, and readability of the post.
        3. Analyze the posts potential for engagement (likes, comments, shares) and its effectiveness in building professional credibility.
        4. Consider the posts relevance to the authors industry, audience, or current trends.
        5. Examine the use of formatting (e.g., line breaks, bullet points), hashtags, mentions, and media (if any).
        6. Evaluate the effectiveness of any call-to-action or takeaway.

        Provide a detailed critique that includes:
        - A brief explanation of the posts strengths and weaknesses.
        - Specific areas that could be improved.
        - Actionable suggestions for enhancing clarity, engagement, and professionalism.

        Your critique will be used to improve the post in the next revision step, so ensure your feedback is thoughtful, constructive, and practical.
        """
    ),
    MessagesPlaceholder(variable_name="messages")
])

##### Creating the reflect chain

In [8]:
reflect_chain=reflection_prompt | llm

##### Initializing MessageGraph

In [10]:
graph=MessageGraph()

C:\Users\Pappu Yadav\AppData\Local\Temp\ipykernel_13360\1934083269.py:1: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph=MessageGraph()


#### Defining the Generation and Reflection node

In [11]:
def generation_node(state:Sequence[BaseMessage]) -> List[BaseMessage]:
    generated_post=generation_chain.invoke({'messages':state})
    return [AIMessage(content=generated_post.content)]

In [14]:
def reflection_node(messages: Sequence[BaseMessage]) -> List[BaseMessage]:
    res=reflect_chain.invoke({"messages":messages})
    return [HumanMessage(content=res.content)]

##### Adding the Generate Node to the Graph

In [15]:
graph.add_node('generate',generation_node)
graph.add_node('reflect',reflection_node)
graph.add_edge('reflect','generate')

##### Set the Entry Point in the Graph

In [16]:
graph.set_entry_point('generate')

##### Adding a Router Node for Decision Making

In [17]:
def should_continue(state: List[BaseMessage]):
    print(state)
    print(len(state))
    print("-"*60)
    if len(state)>6:
        return END
    return 'reflect'


In [18]:
graph.add_conditional_edges('generate',should_continue)

##### Compiling the workflow

In [19]:
workflow=graph.compile()

##### Defining the inputs for the workflow

In [20]:
inputs = HumanMessage(content="""Write a linkedin post on getting a software developer job at IBM under 160 characters""")

##### Executing the workflow

In [21]:
response=workflow.invoke(inputs)

[HumanMessage(content='Write a linkedin post on getting a software developer job at IBM under 160 characters', additional_kwargs={}, response_metadata={}, id='1f39ff97-9889-4286-a24e-2275561e5a6b'), AIMessage(content='🎯 **Final Post (128 characters):**\n\n> Pumped to announce I\'m now a Software Developer at IBM! 🚀 Persistence pays off - keep learning, keep building. #IBM #SoftwareDev\n\n---\n\n**Why this works:**\n\n✅ **Hook** — "Pumped to announce" grabs attention immediately\n✅ **Specific** — Clearly names IBM and the role\n✅ **Emotional** — The 🚀 emoji adds energy without overdoing it\n✅ **Inspirational** — The closing line motivates others on the same journey\n✅ **Hashtags** — Boosts discoverability (kept short to save characters)\n✅ **Length** — Well under the 160-character limit, leaving room for edits\n\n💡 **Pro tip:** If you want even more engagement, consider replacing `#SoftwareDev` with a trending tag like `#IBMJobs` or `#NewGrad` depending on your target audience.', additi

In [26]:
response[-1].content

'# Refined Version (158 characters)\n\n> 🎉 Thrilled to announce I\'ve joined IBM as a Software Developer! 6 months of prep paid off. Grateful to those who believed. What\'s your big career win? 👇 #IBM\n\n---\n\n**What I incorporated from the critique:**\n\n| Feedback | Applied |\n|----------|---------|\n| Add a CTA | ✅ "What\'s your big career win?" invites comments |\n| Be specific | ✅ "6 months of prep" replaces vague "months of hard work" |\n| Add gratitude | ✅ "Grateful to those who believed" |\n| Stronger hashtag | ✅ Swapped `#SoftwareEngineering` for `#IBM` (sharper targeting, less buried) |\n| Under 160 chars | ✅ At 158 chars, still leaves a 2-char buffer |\n\n---\n\n**Why I kept it tight:**\n\nA CTA + gratitude + specificity in 160 characters is already a squeeze — adding the suggested `#SoftwareDev` would have pushed it over. Better to pick *one* sharp hashtag than two average ones.\n\n**The version I\'d *actually* post** (slightly relaxed on the limit):\n\n> 🎉 Thrilled to ann

This table tracks the state transitions in a reflection agent's workflow. Each row represents a step in the process, showing how the state evolves from the initial user input through multiple iterations of generation and reflection. The table captures the iteration number, message type (Human/AI/System), current state, active node (Input/Generate/Reflect), and where the workflow goes next. After 3 iterations, reaching 6 total state changes, the workflow terminates at END.


| Iteration | Type | State | Node | Next Action |
|-----------|------|-------|------|-------------|
| 1 | Human | Initial request | Input | Generate |
| 1 | AI | Generated content | Generate | Reflect |
| 1 | System | Reflection feedback | Reflect | Generate |
| 2 | AI | Revised content | Generate | Reflect |
| 2 | System | Refinement feedback | Reflect | Generate |
| 3 | AI | Final content | Generate | END |
